In [1]:
# Import tools for file handling, data analysis, and evaluation.
from pathlib import Path
import pandas as pd
import numpy as np

# Import the three regression models and evaluation metrics.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Search through the current project for the downloaded monthly files.
files = sorted(Path.cwd().rglob("CRMLSSold*.csv"))

# Exclude the separate 2022–2023 file; use the monthly files for this project.
files = [file for file in files if "20220101_20231231" not in file.name]

if not files:
    raise FileNotFoundError(
        "No CRMLSSold CSV files were found in the project folder."
    )

print("Files found:")
for file in files:
    print(file)

# Load each monthly file and save the filename for documentation.
frames = []
for file in files:
    temp = pd.read_csv(file, low_memory=False)
    temp["source_file"] = file.name
    frames.append(temp)

# Combine the monthly files into one DataFrame.
df = pd.concat(frames, ignore_index=True, sort=False)
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")

# Keep only the required residential single-family properties.
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

Files found:
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202406_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202407_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202409.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202410.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202411.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202412.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202501_filled.csv


In [3]:
# List the predictor columns and the target column.
features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "DaysOnMarket", "ListPrice",
    "Latitude", "Longitude", "City", "CountyOrParish"
]
target = "ClosePrice"

# Convert numeric fields to numeric values. Invalid values become missing.
numeric_features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "DaysOnMarket", "ListPrice",
    "Latitude", "Longitude"
]
categorical_features = ["City", "CountyOrParish"]

for column in numeric_features + [target]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Remove rows without a valid target because the models need a known sale price.
df = df[df[target].notna() & (df[target] > 0)].copy()

# Remove duplicate records with a valid listing ID while preserving rows with no ID.
df = df.sort_values(["CloseDate", "ListingKey"])
with_key = df[df["ListingKey"].notna()].drop_duplicates("ListingKey", keep="last")
without_key = df[df["ListingKey"].isna()]
df = pd.concat([with_key, without_key], ignore_index=True)

# Create a month label for the time-based train/test split.
df["CloseMonth"] = df["CloseDate"].dt.to_period("M").astype(str)
print("Prepared rows:", len(df))
print("Available months:", sorted(df["CloseMonth"].dropna().unique()))

Prepared rows: 78580
Available months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01']


Time-based split
January 2025 is held out as the test set. The six available months immediately before January are used for training, matching the selected window from Week 3. No random split is used because a future month should not be used to predict an earlier month.

In [4]:
# Select the six available months before January 2025 for training.
test_month = "2025-01"
available_training_months = sorted(
    month for month in df["CloseMonth"].dropna().unique()
    if month < test_month
)
train_months = available_training_months[-6:]

train = df[df["CloseMonth"].isin(train_months)].copy()
test = df[df["CloseMonth"] == test_month].copy()

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print("Training months:", train_months)
print("Training rows:", len(train))
print("Test month:", test_month)
print("Test rows:", len(test))

Training months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12']
Training rows: 70436
Test month: 2025-01
Test rows: 8144


In [5]:
# Build the shared preprocessing process.
# Numeric values are median-imputed and standardized.
# Categorical values are filled and one-hot encoded.
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

In [6]:
# Define the models.
# The tree settings limit overfitting while keeping the models useful.
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=20,
        min_samples_leaf=5,
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = []
trained_models = {}

# Train and evaluate every model using the same train/test data.
for name, estimator in models.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model

    results.append({
        "Model": name,
        "R2": r2_score(y_test, predictions),
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions))
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
results_df

,Model,R2,MAE,RMSE
0,Linear Regression,0.651097,77432.497716,1.009587e+06
1,Decision Tree,0.649882,76702.145155,1.011343e+06
2,Random Forest,0.644492,67011.875757,1.019098e+06


In [7]:
# Format the comparison table for easier reading.
comparison = results_df.copy()
comparison["R2"] = comparison["R2"].round(4)
comparison["MAE"] = comparison["MAE"].round(0)
comparison["RMSE"] = comparison["RMSE"].round(0)
display(comparison)

,Model,R2,MAE,RMSE
0,Linear Regression,0.6511,77432.0,1009587.0
1,Decision Tree,0.6499,76702.0,1011343.0
2,Random Forest,0.6445,67012.0,1019098.0


The model with the highest test R2 explains the most variation in January 2025 sale prices among the models tested. MAE is the average absolute dollar error, while RMSE gives more weight to large errors. The best model from this comparison can be carried into later feature-engineering and advanced-modeling weeks.